# 🛍️ B2C Customer Segmentation — Marketo + Salesforce (Commerce Cloud) Integration

## Real-World Scenario
**Company:** GlowHaven — a D2C premium skincare brand selling across its own website, retail partners, and subscription boxes.  
**Challenge:** Marketing runs Marketo for email/SMS campaigns and lifecycle automation. Salesforce Commerce Cloud (B2C Commerce) holds the transactional backbone — orders, returns, product catalog, customer profiles. The VP of Marketing needs:
1. **CLV-based tiers** to allocate retention budgets
2. **Behavioral personas** for personalized campaigns
3. **Churn prediction** signals from engagement decay
4. **Product affinity clusters** for cross-sell recommendations

## Architecture
```
┌──────────────────┐     ┌────────────────────────┐
│     MARKETO       │     │  SALESFORCE B2C COMMERCE│
│  (Lifecycle &     │     │  (Orders, Products,     │
│   Engagement)     │     │   Customer Profiles)    │
└────────┬─────────┘     └────────┬───────────────┘
         │                        │
         ▼                        ▼
   ┌──────────────────────────────────┐
   │  DATA MERGE (Customer Email/ID)  │
   └──────────────┬───────────────────┘
                  ▼
   ┌──────────────────────────────────┐
   │  FEATURE ENGINEERING             │
   │  - Demographics                  │
   │  - Purchase behavior (RFM)       │
   │  - Channel engagement            │
   │  - Product affinity              │
   │  - Customer Lifetime Value       │
   └──────────────┬───────────────────┘
                  ▼
   ┌──────────────────────────────────┐
   │  ADVANCED SEGMENTATION           │
   │  - RFM Scoring                   │
   │  - K-Means Clustering            │
   │  - Gaussian Mixture Models       │
   │  - DBSCAN (Anomaly/VIP detect)   │
   │  - Hierarchical Clustering       │
   │  - CLV-Based Tiering             │
   └──────────────┬───────────────────┘
                  ▼
   ┌──────────────────────────────────┐
   │  PERSONA CREATION & CAMPAIGNS    │
   └──────────────────────────────────┘
```

---

## Step 1 — Environment Setup

In [ ]:
# ============================================================
# STEP 1: Install & Import Libraries
# ============================================================
!pip install -q scikit-learn pandas numpy matplotlib seaborn scipy kneed lifetimes

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# ML & Clustering
from sklearn.preprocessing import StandardScaler, LabelEncoder, MinMaxScaler
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.neighbors import NearestNeighbors
from scipy.cluster.hierarchy import dendrogram, linkage
from kneed import KneeLocator

pd.set_option('display.max_columns', 50)
plt.rcParams['figure.figsize'] = (14, 7)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

print('✅ All libraries loaded.')
print(f'📅 Analysis Date: {datetime.now().strftime("%Y-%m-%d %H:%M")}')

---
## Step 2 — Synthetic Data Generation (Mimicking Real Exports)

### Data Sources:
- **Salesforce Commerce Cloud**: Customer profiles, order history, product catalog  
  *Real extraction*: OCAPI (Open Commerce API) or Business Manager exports
- **Marketo**: Email campaigns, SMS, web tracking, lead scoring  
  *Real extraction*: Marketo REST API `/rest/v1/activities.json`

In [ ]:
# ============================================================
# STEP 2A: Salesforce Commerce Cloud — Customer Profiles
# ============================================================
np.random.seed(42)
N_CUSTOMERS = 15000  # Realistic D2C customer base

# Customer demographics
age_groups = ['18-24', '25-34', '35-44', '45-54', '55-64', '65+']
age_weights = [0.12, 0.28, 0.25, 0.18, 0.11, 0.06]

genders = ['Female', 'Male', 'Non-Binary', 'Prefer not to say']
gender_weights = [0.62, 0.30, 0.05, 0.03]  # Skincare skews female

cities = ['New York', 'Los Angeles', 'Chicago', 'Houston', 'Phoenix',
          'San Francisco', 'Seattle', 'Miami', 'Denver', 'Austin',
          'Portland', 'Boston', 'Dallas', 'Atlanta', 'Nashville']

channels = ['Website Direct', 'Mobile App', 'Instagram Shop', 'Amazon',
            'Retail Partner', 'Subscription Box']
channel_weights = [0.30, 0.22, 0.15, 0.13, 0.12, 0.08]

sf_customer_ids = [f'CUST{str(i).zfill(8)}' for i in range(1, N_CUSTOMERS + 1)]

customers = pd.DataFrame({
    'customer_id': sf_customer_ids,
    'email': [f'user{i}@example.com' for i in range(1, N_CUSTOMERS + 1)],
    'age_group': np.random.choice(age_groups, N_CUSTOMERS, p=age_weights),
    'gender': np.random.choice(genders, N_CUSTOMERS, p=gender_weights),
    'city': np.random.choice(cities, N_CUSTOMERS),
    'acquisition_channel': np.random.choice(channels, N_CUSTOMERS, p=channel_weights),
    'signup_date': pd.NaT,
    'is_subscribed': np.random.choice([True, False], N_CUSTOMERS, p=[0.35, 0.65]),
    'loyalty_tier': np.random.choice(
        ['Bronze', 'Silver', 'Gold', 'Platinum'],
        N_CUSTOMERS, p=[0.45, 0.30, 0.18, 0.07]
    ),
    'has_app_installed': np.random.choice([True, False], N_CUSTOMERS, p=[0.40, 0.60]),
    'preferred_contact': np.random.choice(
        ['Email', 'SMS', 'Push Notification', 'None'],
        N_CUSTOMERS, p=[0.45, 0.25, 0.20, 0.10]
    ),
})

# Signup dates — heavier recent signups (growth company)
base = datetime(2021, 1, 1)
days_range = (datetime(2024, 12, 31) - base).days
signup_days = np.random.beta(2, 1.5, N_CUSTOMERS) * days_range  # Skews recent
customers['signup_date'] = [base + timedelta(days=int(d)) for d in signup_days]

print(f'✅ Customer Profiles: {customers.shape}')
customers.head(3)

In [ ]:
# ============================================================
# STEP 2B: Salesforce Commerce Cloud — Order History
# ============================================================
product_catalog = {
    'Vitamin C Serum': {'category': 'Serums', 'price': 48.00, 'cogs': 12.00},
    'Retinol Night Cream': {'category': 'Moisturizers', 'price': 62.00, 'cogs': 15.50},
    'Hyaluronic Acid Toner': {'category': 'Toners', 'price': 34.00, 'cogs': 8.50},
    'SPF 50 Daily Shield': {'category': 'Sunscreen', 'price': 38.00, 'cogs': 9.50},
    'Gentle Foam Cleanser': {'category': 'Cleansers', 'price': 28.00, 'cogs': 7.00},
    'AHA/BHA Exfoliant': {'category': 'Exfoliants', 'price': 42.00, 'cogs': 10.50},
    'Niacinamide Booster': {'category': 'Serums', 'price': 36.00, 'cogs': 9.00},
    'Eye Repair Complex': {'category': 'Eye Care', 'price': 55.00, 'cogs': 13.75},
    'Overnight Mask': {'category': 'Masks', 'price': 45.00, 'cogs': 11.25},
    'Complete Routine Kit': {'category': 'Kits', 'price': 149.00, 'cogs': 37.25},
    'Travel Essentials Set': {'category': 'Kits', 'price': 68.00, 'cogs': 17.00},
    'Lip Treatment Balm': {'category': 'Lip Care', 'price': 22.00, 'cogs': 5.50},
}

orders = []
for _, cust in customers.iterrows():
    # Order frequency varies by loyalty tier
    tier = cust['loyalty_tier']
    avg_orders = {'Bronze': 2, 'Silver': 4, 'Gold': 7, 'Platinum': 12}[tier]
    n_orders = max(1, int(np.random.poisson(avg_orders)))

    for _ in range(n_orders):
        order_date = cust['signup_date'] + timedelta(
            days=int(np.random.uniform(1, max(2, (datetime(2024,12,31) - cust['signup_date']).days)))
        )
        if order_date > datetime(2024, 12, 31):
            order_date = datetime(2024, 12, 31) - timedelta(days=np.random.randint(1, 30))

        # Items per order
        n_items = np.random.choice([1, 2, 3, 4, 5], p=[0.35, 0.30, 0.20, 0.10, 0.05])
        ordered_products = np.random.choice(list(product_catalog.keys()), n_items, replace=False)

        for product in ordered_products:
            info = product_catalog[product]
            qty = np.random.choice([1, 2, 3], p=[0.70, 0.22, 0.08])
            discount = np.random.choice([0, 0.10, 0.15, 0.20, 0.25], p=[0.50, 0.20, 0.15, 0.10, 0.05])
            orders.append({
                'customer_id': cust['customer_id'],
                'order_id': f'ORD{np.random.randint(10000000, 99999999)}',
                'order_date': order_date,
                'product_name': product,
                'product_category': info['category'],
                'unit_price': info['price'],
                'cogs': info['cogs'],
                'quantity': qty,
                'discount_pct': discount,
                'channel': np.random.choice(channels, p=channel_weights),
                'is_return': np.random.choice([True, False], p=[0.06, 0.94]),
                'payment_method': np.random.choice(
                    ['Credit Card', 'PayPal', 'Apple Pay', 'Klarna', 'Gift Card'],
                    p=[0.40, 0.20, 0.20, 0.12, 0.08]
                ),
            })

orders_df = pd.DataFrame(orders)
orders_df['order_date'] = pd.to_datetime(orders_df['order_date'])
orders_df['line_total'] = (orders_df['unit_price'] * orders_df['quantity'] *
                           (1 - orders_df['discount_pct'])).round(2)
orders_df['line_profit'] = (
    orders_df['line_total'] - orders_df['cogs'] * orders_df['quantity']
).round(2)

print(f'✅ Order History: {orders_df.shape}')
print(f'   Total Revenue: ${orders_df["line_total"].sum():,.0f}')
print(f'   Unique Orders: {orders_df["order_id"].nunique():,}')
orders_df.head(3)

In [ ]:
# ============================================================
# STEP 2C: Marketo Engagement Data
# ============================================================
mkto_b2c = []
b2c_activity_types = [
    'Email Opened', 'Email Clicked', 'Email Bounced', 'SMS Delivered',
    'SMS Clicked', 'Push Notification Opened', 'Web Page Visited',
    'Cart Abandoned', 'Wishlist Added', 'Review Submitted',
    'Referral Sent', 'Loyalty Points Redeemed', 'Unsubscribed'
]
b2c_activity_weights = [0.22, 0.10, 0.02, 0.12, 0.05, 0.08,
                        0.18, 0.07, 0.05, 0.03, 0.03, 0.03, 0.02]

b2c_campaigns = [
    'Welcome_Series', 'Abandoned_Cart', 'Post_Purchase', 'Birthday_Offer',
    'VIP_Early_Access', 'Seasonal_Sale', 'Product_Launch',
    'Refer_A_Friend', 'Re-engagement_30day', 'Loyalty_Milestone',
    'Skincare_Quiz', 'Subscription_Upsell', 'Review_Request'
]

for _, cust in customers.iterrows():
    n_activities = max(1, int(np.random.poisson(12)))
    for _ in range(n_activities):
        act_date = cust['signup_date'] + timedelta(
            days=int(np.random.uniform(0, max(1, (datetime(2024,12,31) - cust['signup_date']).days)))
        )
        mkto_b2c.append({
            'customer_id': cust['customer_id'],
            'activity_type': np.random.choice(b2c_activity_types, p=b2c_activity_weights),
            'activity_date': act_date,
            'campaign': np.random.choice(b2c_campaigns),
            'device': np.random.choice(['Mobile', 'Desktop', 'Tablet'], p=[0.55, 0.35, 0.10]),
            'engagement_score': np.random.randint(1, 100),
        })

mkto_b2c_df = pd.DataFrame(mkto_b2c)
mkto_b2c_df['activity_date'] = pd.to_datetime(mkto_b2c_df['activity_date'])

print(f'✅ Marketo B2C Activities: {mkto_b2c_df.shape}')
print(f'   Activity types: {mkto_b2c_df["activity_type"].nunique()}')
mkto_b2c_df.head(3)

---
## Step 3 — Data Validation & Quality Checks

In [ ]:
# ============================================================
# STEP 3: Data Validation
# ============================================================
def validate(df, name, key):
    print(f'\n{"=" * 60}')
    print(f'📊 VALIDATION: {name}')
    print(f'{"=" * 60}')
    print(f'Shape: {df.shape[0]:,} × {df.shape[1]}')
    print(f'Memory: {df.memory_usage(deep=True).sum()/1024**2:.1f} MB')
    missing = df.isnull().sum()
    if missing.any():
        print(f'Missing: {missing[missing > 0].to_dict()}')
    else:
        print('Missing: ✅ None')
    print(f'Key [{key}]: {df[key].nunique():,} unique, {df[key].duplicated().sum():,} dupes')
    nums = df.select_dtypes(include=[np.number]).columns
    if len(nums) > 0:
        print(f'Numeric range checks:')
        for c in nums[:6]:
            print(f'  {c}: [{df[c].min():.2f}, {df[c].max():.2f}] μ={df[c].mean():.2f}')

validate(customers, 'Customer Profiles (SFCC)', 'customer_id')
validate(orders_df, 'Order History (SFCC)', 'customer_id')
validate(mkto_b2c_df, 'Marketo Engagement (B2C)', 'customer_id')

In [ ]:
# ============================================================
# STEP 3B: Join Key Check & Variable Catalog
# ============================================================
cust_ids = set(customers['customer_id'])
order_ids = set(orders_df['customer_id'])
mkto_ids = set(mkto_b2c_df['customer_id'])

print('🔗 JOIN KEY INTEGRITY')
print(f'  Customers:       {len(cust_ids):,}')
print(f'  In Orders:       {len(order_ids):,} ({len(order_ids & cust_ids)/len(cust_ids)*100:.1f}% match)')
print(f'  In Marketo:      {len(mkto_ids):,} ({len(mkto_ids & cust_ids)/len(cust_ids)*100:.1f}% match)')
print(f'  All three:       {len(cust_ids & order_ids & mkto_ids):,}')

# Variable catalog
catalog = pd.DataFrame([
    ['customer_id', 'Both', 'Unique customer ID', 'String', 'Key'],
    ['age_group', 'SFCC', 'Age band', 'Ordinal', 'Demographic'],
    ['gender', 'SFCC', 'Gender identity', 'Categorical', 'Demographic'],
    ['city', 'SFCC', 'City of residence', 'Categorical', 'Geographic'],
    ['acquisition_channel', 'SFCC', 'First-touch channel', 'Categorical', 'Attribution'],
    ['loyalty_tier', 'SFCC', 'Bronze/Silver/Gold/Platinum', 'Ordinal', 'Loyalty'],
    ['order_date', 'SFCC', 'Transaction timestamp', 'DateTime', 'Transaction'],
    ['product_name', 'SFCC', 'SKU purchased', 'Categorical', 'Product'],
    ['line_total', 'SFCC', 'Net order line revenue', 'Continuous', 'Financial'],
    ['activity_type', 'Marketo', 'Engagement event type', 'Categorical', 'Engagement'],
    ['campaign', 'Marketo', 'Campaign name', 'Categorical', 'Engagement'],
    ['device', 'Marketo', 'Device type', 'Categorical', 'Behavioral'],
    ['engagement_score', 'Marketo', 'Marketo engagement score', 'Continuous', 'Engagement'],
], columns=['Variable', 'Source', 'Description', 'Type', 'Category'])
print('\n📋 VARIABLE CATALOG')
print(catalog.to_string(index=False))

---
## Step 4 — Feature Engineering

In [ ]:
# ============================================================
# STEP 4A: Transaction Feature Rollup
# ============================================================
ref_date = datetime(2024, 12, 31)

txn_features = orders_df.groupby('customer_id').agg(
    total_orders=('order_id', 'nunique'),
    total_items=('quantity', 'sum'),
    total_revenue=('line_total', 'sum'),
    total_profit=('line_profit', 'sum'),
    avg_order_value=('line_total', 'mean'),
    max_order_value=('line_total', 'max'),
    avg_items_per_order=('quantity', 'mean'),
    unique_products=('product_name', 'nunique'),
    unique_categories=('product_category', 'nunique'),
    return_count=('is_return', 'sum'),
    avg_discount=('discount_pct', 'mean'),
    first_order_date=('order_date', 'min'),
    last_order_date=('order_date', 'max'),
    unique_channels=('channel', 'nunique'),
).reset_index()

# Derived metrics
txn_features['days_since_last_order'] = (ref_date - txn_features['last_order_date']).dt.days
txn_features['customer_tenure_days'] = (ref_date - txn_features['first_order_date']).dt.days
txn_features['order_frequency'] = (
    txn_features['total_orders'] / (txn_features['customer_tenure_days'] / 30).clip(lower=1)
).round(3)
txn_features['return_rate'] = (txn_features['return_count'] / txn_features['total_orders'] * 100).round(2)
txn_features['profit_margin'] = (txn_features['total_profit'] / txn_features['total_revenue'].clip(lower=1) * 100).round(2)
txn_features['revenue_per_day'] = (txn_features['total_revenue'] / txn_features['customer_tenure_days'].clip(lower=1)).round(4)

print(f'✅ Transaction features: {txn_features.shape}')
txn_features.head(3)

In [ ]:
# ============================================================
# STEP 4B: Product Affinity Features
# ============================================================
# Percentage of spend by category
cat_spend = orders_df.groupby(['customer_id', 'product_category'])['line_total'].sum().unstack(fill_value=0)
cat_spend_pct = cat_spend.div(cat_spend.sum(axis=1), axis=0).round(4)
cat_spend_pct.columns = [f'pct_spend_{c.lower().replace(" ", "_")}' for c in cat_spend_pct.columns]
cat_spend_pct = cat_spend_pct.reset_index()

# Favorite product category
fav_cat = orders_df.groupby('customer_id')['product_category'].agg(
    lambda x: x.value_counts().index[0]
).reset_index()
fav_cat.columns = ['customer_id', 'favorite_category']

print(f'✅ Product affinity features: {cat_spend_pct.shape[1] - 1} categories')

In [ ]:
# ============================================================
# STEP 4C: Marketo Engagement Rollup
# ============================================================
# Activity counts by type
act_pivot = mkto_b2c_df.groupby(['customer_id', 'activity_type']).size().unstack(fill_value=0)
act_pivot.columns = [c.lower().replace(' ', '_') + '_count' for c in act_pivot.columns]
act_pivot = act_pivot.reset_index()

# Engagement rollup
eng_features = mkto_b2c_df.groupby('customer_id').agg(
    total_engagements=('activity_type', 'count'),
    avg_engagement_score=('engagement_score', 'mean'),
    max_engagement_score=('engagement_score', 'max'),
    unique_campaigns=('campaign', 'nunique'),
    last_engagement_date=('activity_date', 'max'),
    first_engagement_date=('activity_date', 'min'),
    primary_device=('device', lambda x: x.value_counts().index[0]),
    mobile_pct=('device', lambda x: (x == 'Mobile').sum() / len(x) * 100),
).reset_index()

eng_features['days_since_last_engagement'] = (ref_date - eng_features['last_engagement_date']).dt.days
eng_features['engagement_span_days'] = (eng_features['last_engagement_date'] -
                                         eng_features['first_engagement_date']).dt.days

# Merge engagement activity counts
eng_full = eng_features.merge(act_pivot, on='customer_id', how='left').fillna(0)

print(f'✅ Engagement features: {eng_full.shape}')

In [ ]:
# ============================================================
# STEP 4D: Customer Lifetime Value (CLV) Estimation
# ============================================================
# Simplified CLV = (Avg Monthly Revenue) × (Expected Lifetime Months) × Profit Margin
clv_data = txn_features[['customer_id', 'total_revenue', 'total_profit',
                          'customer_tenure_days', 'total_orders', 'profit_margin']].copy()

clv_data['monthly_revenue'] = clv_data['total_revenue'] / (clv_data['customer_tenure_days'] / 30).clip(lower=1)
clv_data['monthly_orders'] = clv_data['total_orders'] / (clv_data['customer_tenure_days'] / 30).clip(lower=1)

# Expected lifetime: higher for more frequent/recent buyers
# Simple heuristic: if monthly orders > median, expect 24 months; else 12
median_monthly = clv_data['monthly_orders'].median()
clv_data['expected_lifetime_months'] = np.where(
    clv_data['monthly_orders'] > median_monthly, 24, 12
)

clv_data['estimated_clv'] = (
    clv_data['monthly_revenue'] *
    clv_data['expected_lifetime_months'] *
    clv_data['profit_margin'] / 100
).round(2)

# CLV Tiers
clv_data['clv_tier'] = pd.qcut(clv_data['estimated_clv'], q=5,
    labels=['Bottom 20%', 'Low', 'Mid', 'High', 'Top 20%'])

print('📊 CLV DISTRIBUTION')
print(clv_data['estimated_clv'].describe().round(2).to_string())
print(f'\n--- CLV Tiers ---')
print(clv_data.groupby('clv_tier')['estimated_clv'].agg(['count','mean','sum']).round(0).to_string())

In [ ]:
# ============================================================
# STEP 4E: Build Unified Feature Matrix
# ============================================================
unified = customers.merge(txn_features, on='customer_id', how='left')
unified = unified.merge(eng_full, on='customer_id', how='left')
unified = unified.merge(cat_spend_pct, on='customer_id', how='left')
unified = unified.merge(fav_cat, on='customer_id', how='left')
unified = unified.merge(
    clv_data[['customer_id', 'estimated_clv', 'clv_tier', 'monthly_revenue']],
    on='customer_id', how='left'
)

# Fill NaN
num_cols = unified.select_dtypes(include=[np.number]).columns
unified[num_cols] = unified[num_cols].fillna(0)

# Encode categoricals
le = LabelEncoder()
for col in ['age_group', 'gender', 'acquisition_channel', 'loyalty_tier', 'favorite_category']:
    if col in unified.columns:
        unified[col + '_encoded'] = le.fit_transform(unified[col].astype(str))

# Composite scores
unified['engagement_index'] = (
    unified['avg_engagement_score'] * 0.3 +
    np.log1p(unified['total_engagements']) * 10 * 0.3 +
    np.clip(100 - unified['days_since_last_engagement'], 0, 100) * 0.2 +
    unified['unique_campaigns'] * 5 * 0.2
).clip(0, 100).round(2)

unified['purchase_intensity'] = (
    np.log1p(unified['total_revenue']) * 5 * 0.4 +
    unified['order_frequency'] * 30 * 0.3 +
    unified['unique_products'] * 5 * 0.3
).clip(0, 100).round(2)

print(f'\n✅ Unified B2C Feature Matrix: {unified.shape}')
print(f'   Total features: {unified.shape[1]}')
unified.head(3)

In [ ]:
# ============================================================
# STEP 4F: Select & Scale Clustering Features
# ============================================================
cluster_features = [
    # Purchase behavior
    'total_orders', 'total_revenue', 'avg_order_value', 'order_frequency',
    'unique_products', 'return_rate', 'days_since_last_order',
    # Engagement
    'total_engagements', 'avg_engagement_score', 'engagement_index',
    'days_since_last_engagement', 'unique_campaigns', 'mobile_pct',
    # Value
    'estimated_clv', 'purchase_intensity', 'monthly_revenue',
    'profit_margin', 'avg_discount',
    # Diversity
    'unique_categories', 'unique_channels',
]

X_b2c = unified[cluster_features].copy()

# Log-transform skewed
skewed = ['total_revenue', 'estimated_clv', 'avg_order_value', 'monthly_revenue']
for col in skewed:
    X_b2c[col] = np.log1p(X_b2c[col])

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_b2c)

print(f'✅ Scaled feature matrix: {X_scaled.shape}')

---
## Step 5 — Exploratory Data Analysis

In [ ]:
# ============================================================
# STEP 5A: Distribution of Key B2C Features
# ============================================================
fig, axes = plt.subplots(3, 3, figsize=(18, 14))
plot_feats = ['total_revenue', 'total_orders', 'avg_order_value',
              'estimated_clv', 'engagement_index', 'purchase_intensity',
              'order_frequency', 'days_since_last_order', 'return_rate']

for ax, feat in zip(axes.flat, plot_feats):
    data = unified[feat].clip(upper=unified[feat].quantile(0.99))
    ax.hist(data, bins=50, edgecolor='white', alpha=0.7, color='#7C3AED')
    ax.set_title(feat.replace('_', ' ').title(), fontsize=11, fontweight='bold')
    ax.axvline(data.median(), color='red', linestyle='--', alpha=0.7)

plt.suptitle('B2C Customer Feature Distributions', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# STEP 5B: Correlation & Demographic Breakdowns
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Correlation
corr = unified[cluster_features].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='RdBu_r', center=0, ax=axes[0],
            annot=False, linewidths=0.3, vmin=-1, vmax=1)
axes[0].set_title('Feature Correlation Matrix', fontweight='bold')

# Revenue by acquisition channel
ch_rev = unified.groupby('acquisition_channel')['total_revenue'].mean().sort_values()
ch_rev.plot.barh(ax=axes[1], color='#7C3AED', edgecolor='white')
axes[1].set_title('Avg Revenue by Acquisition Channel', fontweight='bold')
axes[1].set_xlabel('Avg Revenue ($)')

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# STEP 5C: CLV Distribution by Demographics
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

unified.boxplot(column='estimated_clv', by='age_group', ax=axes[0],
                patch_artist=True, showfliers=False)
axes[0].set_title('CLV by Age Group', fontweight='bold')
axes[0].set_xlabel(''); plt.sca(axes[0]); plt.title('')

unified.boxplot(column='estimated_clv', by='loyalty_tier', ax=axes[1],
                patch_artist=True, showfliers=False)
axes[1].set_title('CLV by Loyalty Tier', fontweight='bold')
axes[1].set_xlabel(''); plt.sca(axes[1]); plt.title('')

unified.boxplot(column='engagement_index', by='loyalty_tier', ax=axes[2],
                patch_artist=True, showfliers=False)
axes[2].set_title('Engagement by Loyalty Tier', fontweight='bold')
axes[2].set_xlabel(''); plt.sca(axes[2]); plt.title('')

plt.suptitle('B2C Customer Deep Dive', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## Step 6 — Advanced Segmentation

### 6A: RFM Analysis

In [ ]:
# ============================================================
# STEP 6A: B2C RFM Segmentation
# ============================================================
rfm = unified[['customer_id', 'days_since_last_order',
               'total_orders', 'total_revenue']].copy()
rfm.columns = ['customer_id', 'Recency', 'Frequency', 'Monetary']

# Handle zero-purchase customers
rfm['Recency'] = rfm['Recency'].replace(0, rfm['Recency'].max())
rfm['Frequency'] = rfm['Frequency'].clip(lower=1)

rfm['R'] = pd.qcut(rfm['Recency'], 5, labels=[5,4,3,2,1]).astype(int)
rfm['F'] = pd.qcut(rfm['Frequency'].rank(method='first'), 5, labels=[1,2,3,4,5]).astype(int)
rfm['M'] = pd.qcut(rfm['Monetary'].rank(method='first'), 5, labels=[1,2,3,4,5]).astype(int)
rfm['RFM_Score'] = rfm['R'] + rfm['F'] + rfm['M']

def b2c_segment(row):
    r, f, m = row['R'], row['F'], row['M']
    if r >= 4 and f >= 4 and m >= 4:
        return '💎 VIP / Loyalists'
    elif r >= 4 and f >= 3 and m >= 3:
        return '⭐ Potential Loyalists'
    elif r >= 4 and f < 3:
        return '🆕 Recent Converts'
    elif r >= 3 and m >= 4:
        return '💰 Big Spenders'
    elif r < 3 and f >= 4:
        return '⚠️ At-Risk Loyalists'
    elif r < 3 and f >= 3 and m >= 3:
        return '🔄 Need Attention'
    elif r < 2 and f < 2:
        return '💤 Hibernating'
    elif r < 3 and f < 3:
        return '👋 About to Sleep'
    else:
        return '🌱 Promising'

rfm['RFM_Segment'] = rfm.apply(b2c_segment, axis=1)

print('📊 B2C RFM SEGMENTS')
rfm_summary = rfm.groupby('RFM_Segment').agg(
    count=('customer_id', 'count'),
    avg_recency=('Recency', 'mean'),
    avg_frequency=('Frequency', 'mean'),
    avg_monetary=('Monetary', 'mean'),
).round(1)
rfm_summary['pct'] = (rfm_summary['count'] / len(rfm) * 100).round(1)
print(rfm_summary.sort_values('avg_monetary', ascending=False).to_string())

In [ ]:
# RFM Visualization
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

seg_counts = rfm['RFM_Segment'].value_counts()
colors = plt.cm.Set3(np.linspace(0, 1, len(seg_counts)))
seg_counts.plot.barh(ax=axes[0], color=colors, edgecolor='white')
axes[0].set_title('B2C RFM Segments', fontweight='bold')
for i, v in enumerate(seg_counts.values):
    axes[0].text(v + 30, i, f'{v:,} ({v/len(rfm)*100:.1f}%)', va='center', fontsize=9)

# Revenue contribution per segment
seg_rev = rfm.groupby('RFM_Segment')['Monetary'].sum().sort_values()
seg_rev_pct = (seg_rev / seg_rev.sum() * 100)
seg_rev_pct.plot.barh(ax=axes[1], color='#7C3AED', edgecolor='white')
axes[1].set_title('Revenue Contribution by Segment (%)', fontweight='bold')
axes[1].set_xlabel('% of Total Revenue')

plt.tight_layout()
plt.show()

### 6B: K-Means with Optimal K

In [ ]:
# ============================================================
# STEP 6B: K-Means Clustering
# ============================================================
K_range = range(2, 12)
metrics = {'inertia': [], 'silhouette': [], 'calinski': [], 'davies': []}

for k in K_range:
    km = KMeans(n_clusters=k, init='k-means++', n_init=15, random_state=42)
    labs = km.fit_predict(X_scaled)
    metrics['inertia'].append(km.inertia_)
    metrics['silhouette'].append(silhouette_score(X_scaled, labs))
    metrics['calinski'].append(calinski_harabasz_score(X_scaled, labs))
    metrics['davies'].append(davies_bouldin_score(X_scaled, labs))

kn = KneeLocator(list(K_range), metrics['inertia'], curve='convex', direction='decreasing')
elbow_k = kn.knee if kn.knee else 5
sil_k = list(K_range)[np.argmax(metrics['silhouette'])]

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
titles = ['Elbow (Inertia)', 'Silhouette', 'Calinski-Harabasz', 'Davies-Bouldin']
colors_m = ['#2563EB', '#10B981', '#F59E0B', '#EF4444']
for ax, (key, vals), title, c in zip(axes.flat, metrics.items(), titles, colors_m):
    ax.plot(K_range, vals, 'o-', color=c, linewidth=2)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('K')

axes[0,0].axvline(elbow_k, color='red', linestyle='--', label=f'Elbow K={elbow_k}')
axes[0,0].legend()
axes[0,1].axvline(sil_k, color='red', linestyle='--', label=f'Best K={sil_k}')
axes[0,1].legend()

plt.suptitle('B2C Optimal K Selection', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

FINAL_K = max(elbow_k, sil_k)
if FINAL_K > 7:
    FINAL_K = 6
print(f'🎯 Selected K = {FINAL_K}')

In [ ]:
# Fit K-Means
kmeans = KMeans(n_clusters=FINAL_K, init='k-means++', n_init=25, random_state=42)
unified['kmeans_cluster'] = kmeans.fit_predict(X_scaled)

# PCA for visualization
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

fig, ax = plt.subplots(figsize=(14, 8))
scatter = ax.scatter(X_pca[:, 0], X_pca[:, 1],
    c=unified['kmeans_cluster'], cmap='Set1', s=8, alpha=0.4)
centroids_pca = pca.transform(kmeans.cluster_centers_)
ax.scatter(centroids_pca[:, 0], centroids_pca[:, 1],
    c='black', marker='X', s=250, linewidth=2, edgecolors='white', zorder=5)
ax.set_title(f'B2C K-Means Clusters (K={FINAL_K}) — PCA', fontweight='bold', fontsize=14)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
plt.colorbar(scatter, label='Cluster')
plt.tight_layout()
plt.show()

print(f'Silhouette: {silhouette_score(X_scaled, unified["kmeans_cluster"]):.4f}')

### 6C: Gaussian Mixture Model

In [ ]:
# ============================================================
# STEP 6C: GMM — Soft Clustering with Probability
# ============================================================
bic = []
for k in range(2, 10):
    g = GaussianMixture(n_components=k, covariance_type='full', random_state=42, n_init=5)
    g.fit(X_scaled)
    bic.append(g.bic(X_scaled))

best_gmm_k = range(2, 10)[np.argmin(bic)]

gmm = GaussianMixture(n_components=FINAL_K, covariance_type='full', random_state=42, n_init=10)
unified['gmm_cluster'] = gmm.fit_predict(X_scaled)
unified['gmm_confidence'] = gmm.predict_proba(X_scaled).max(axis=1).round(4)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].plot(range(2, 10), bic, 'o-', color='#7C3AED', linewidth=2)
axes[0].axvline(best_gmm_k, color='red', linestyle='--', label=f'Best BIC K={best_gmm_k}')
axes[0].set_title('GMM — BIC Score', fontweight='bold')
axes[0].legend()

axes[1].hist(unified['gmm_confidence'], bins=50, color='#7C3AED', edgecolor='white', alpha=0.7)
axes[1].axvline(0.5, color='red', linestyle='--')
axes[1].set_title('GMM Cluster Assignment Confidence', fontweight='bold')
axes[1].set_xlabel('Max Probability')

plt.tight_layout()
plt.show()

print(f'Low-confidence customers (<0.5): {(unified["gmm_confidence"] < 0.5).sum():,}')
print(f'These are "swing" customers at cluster boundaries — ideal for targeted testing.')

### 6D: DBSCAN — VIP & Anomaly Detection

In [ ]:
# ============================================================
# STEP 6D: DBSCAN for VIP / Outlier Detection
# ============================================================
pca5 = PCA(n_components=5, random_state=42)
X_pca5 = pca5.fit_transform(X_scaled)

nn = NearestNeighbors(n_neighbors=10)
nn.fit(X_pca5)
dists, _ = nn.kneighbors(X_pca5)
sorted_d = np.sort(dists[:, -1])

kn_eps = KneeLocator(range(len(sorted_d)), sorted_d, curve='convex', direction='increasing')
eps_val = sorted_d[kn_eps.knee] if kn_eps.knee else 3.0

db = DBSCAN(eps=eps_val, min_samples=15)
unified['dbscan_cluster'] = db.fit_predict(X_pca5)

n_clusters = len(set(unified['dbscan_cluster'])) - (1 if -1 in unified['dbscan_cluster'].values else 0)
n_outliers = (unified['dbscan_cluster'] == -1).sum()

# Outliers are often VIPs or fraudulent accounts
outlier_profile = unified[unified['dbscan_cluster'] == -1][[
    'total_revenue', 'total_orders', 'estimated_clv', 'engagement_index'
]].describe().round(1)

print(f'DBSCAN: {n_clusters} clusters, {n_outliers} outliers ({n_outliers/len(unified)*100:.1f}%)')
print(f'\n--- Outlier Profile (potential VIPs or anomalies) ---')
print(outlier_profile.to_string())

# Visualize
fig, ax = plt.subplots(figsize=(14, 8))
main = unified['dbscan_cluster'] != -1
ax.scatter(X_pca[main, 0], X_pca[main, 1], c=unified.loc[main, 'dbscan_cluster'],
           cmap='Set2', s=8, alpha=0.3, label='Core')
ax.scatter(X_pca[~main, 0], X_pca[~main, 1], c='red', marker='x', s=25,
           alpha=0.6, label=f'Outliers ({n_outliers})')
ax.set_title('DBSCAN — Density Clusters & Outliers', fontweight='bold')
ax.legend(fontsize=12)
plt.tight_layout()
plt.show()

### 6E: Hierarchical Clustering

In [ ]:
# ============================================================
# STEP 6E: Hierarchical Clustering
# ============================================================
sample_idx = np.random.choice(len(X_scaled), size=min(800, len(X_scaled)), replace=False)
linkage_mat = linkage(X_scaled[sample_idx], method='ward')

fig, ax = plt.subplots(figsize=(18, 8))
dendrogram(linkage_mat, truncate_mode='lastp', p=30,
           leaf_rotation=90, ax=ax,
           color_threshold=linkage_mat[-FINAL_K+1, 2])
ax.axhline(linkage_mat[-FINAL_K+1, 2], color='red', linestyle='--',
           label=f'Cut → {FINAL_K} clusters')
ax.set_title('B2C Hierarchical Dendrogram (Ward)', fontsize=14, fontweight='bold')
ax.legend(fontsize=12)
plt.tight_layout()
plt.show()

hc = AgglomerativeClustering(n_clusters=FINAL_K, linkage='ward')
unified['hc_cluster'] = hc.fit_predict(X_scaled)

print(f'Hierarchical Silhouette: {silhouette_score(X_scaled, unified["hc_cluster"]):.4f}')

---
## Step 7 — Model Comparison

In [ ]:
# ============================================================
# STEP 7: Compare All Methods
# ============================================================
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

methods = ['kmeans_cluster', 'gmm_cluster', 'hc_cluster']
names = ['K-Means', 'GMM', 'Hierarchical']

print('📊 B2C CLUSTERING COMPARISON')
print('=' * 70)
for name, col in zip(names, methods):
    s = silhouette_score(X_scaled, unified[col])
    c = calinski_harabasz_score(X_scaled, unified[col])
    d = davies_bouldin_score(X_scaled, unified[col])
    print(f'  {name:15s} | Silhouette: {s:.4f} | Calinski: {c:,.0f} | Davies-Bouldin: {d:.4f}')

print(f'\n--- Agreement ---')
for i in range(len(methods)):
    for j in range(i+1, len(methods)):
        ari = adjusted_rand_score(unified[methods[i]], unified[methods[j]])
        nmi = normalized_mutual_info_score(unified[methods[i]], unified[methods[j]])
        print(f'  {names[i]} vs {names[j]}: ARI={ari:.4f}, NMI={nmi:.4f}')

# Visual comparison
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
for ax, col, name in zip(axes, methods, names):
    sc = ax.scatter(X_pca[:, 0], X_pca[:, 1], c=unified[col], cmap='Set1', s=6, alpha=0.4)
    ax.set_title(name, fontweight='bold', fontsize=13)
    plt.colorbar(sc, ax=ax, shrink=0.7)
plt.suptitle('B2C Method Comparison', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Step 8 — Persona Creation & Segment Profiling

In [ ]:
# ============================================================
# STEP 8A: Deep Segment Profiling
# ============================================================
profile_cols = [
    'total_revenue', 'total_orders', 'avg_order_value', 'estimated_clv',
    'order_frequency', 'engagement_index', 'purchase_intensity',
    'days_since_last_order', 'unique_products', 'return_rate',
    'avg_engagement_score', 'mobile_pct', 'avg_discount'
]

profile = unified.groupby('kmeans_cluster')[profile_cols].mean().round(2)
profile['count'] = unified.groupby('kmeans_cluster').size()
profile['pct'] = (profile['count'] / len(unified) * 100).round(1)
profile['total_segment_revenue'] = unified.groupby('kmeans_cluster')['total_revenue'].sum().round(0)
profile['revenue_share_pct'] = (
    profile['total_segment_revenue'] / profile['total_segment_revenue'].sum() * 100
).round(1)

print('📊 B2C SEGMENT PROFILES')
print(profile[['count', 'pct', 'total_revenue', 'estimated_clv',
               'engagement_index', 'order_frequency', 'revenue_share_pct']].to_string())

In [ ]:
# ============================================================
# STEP 8B: Assign Persona Names
# ============================================================
cluster_rank = unified.groupby('kmeans_cluster')['estimated_clv'].mean().sort_values(ascending=False)
order = cluster_rank.index.tolist()

persona_map = {}
persona_labels = [
    '💎 VIP Skincare Enthusiasts',
    '⭐ Loyal Routine Buyers',
    '🛒 Deal-Seeking Explorers',
    '🌱 Casual Newcomers',
    '💤 Lapsed / Dormant',
    '🔬 Niche Product Fans',
]
for i, cid in enumerate(order):
    persona_map[cid] = persona_labels[i] if i < len(persona_labels) else f'Segment {i+1}'

unified['persona'] = unified['kmeans_cluster'].map(persona_map)

print('\n🏷️ B2C PERSONA ASSIGNMENT')
for cid, name in sorted(persona_map.items()):
    mask = unified['kmeans_cluster'] == cid
    n = mask.sum()
    rev = unified.loc[mask, 'total_revenue'].mean()
    clv = unified.loc[mask, 'estimated_clv'].mean()
    eng = unified.loc[mask, 'engagement_index'].mean()
    print(f'  {name}: {n:,} customers | Avg Rev: ${rev:,.0f} | CLV: ${clv:,.0f} | Eng: {eng:.1f}')

In [ ]:
# ============================================================
# STEP 8C: Radar Chart — Persona Comparison
# ============================================================
radar_feats = ['engagement_index', 'purchase_intensity', 'order_frequency',
               'avg_order_value', 'estimated_clv']

radar_data = unified.groupby('kmeans_cluster')[radar_feats].mean()
radar_n = (radar_data - radar_data.min()) / (radar_data.max() - radar_data.min())

angles = np.linspace(0, 2 * np.pi, len(radar_feats), endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(polar=True))
colors = plt.cm.Set2(np.linspace(0, 0.9, FINAL_K))

for idx, (cid, row) in enumerate(radar_n.iterrows()):
    vals = row.tolist() + [row.tolist()[0]]
    ax.plot(angles, vals, 'o-', linewidth=2, color=colors[idx],
            label=persona_map.get(cid, f'C{cid}'))
    ax.fill(angles, vals, alpha=0.1, color=colors[idx])

ax.set_xticks(angles[:-1])
ax.set_xticklabels([f.replace('_', '\n').title() for f in radar_feats], fontsize=10)
ax.set_title('B2C Persona Radar', fontweight='bold', fontsize=14, pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.4, 1.1), fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# STEP 8D: Persona × Demographics Cross-Analysis
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Age breakdown
ct_age = pd.crosstab(unified['persona'], unified['age_group'], normalize='index') * 100
ct_age.plot.bar(stacked=True, ax=axes[0], colormap='Set3', edgecolor='white')
axes[0].set_title('Persona × Age Group', fontweight='bold')
axes[0].legend(title='Age', fontsize=8, loc='upper right')
axes[0].tick_params(axis='x', rotation=30)

# Channel breakdown
ct_ch = pd.crosstab(unified['persona'], unified['acquisition_channel'], normalize='index') * 100
ct_ch.plot.bar(stacked=True, ax=axes[1], colormap='Set2', edgecolor='white')
axes[1].set_title('Persona × Acquisition Channel', fontweight='bold')
axes[1].legend(title='Channel', fontsize=7, loc='upper right')
axes[1].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

---
## Step 9 — Campaign Strategy & Export

In [ ]:
# ============================================================
# STEP 9A: Campaign Playbook by Persona
# ============================================================
playbook = {
    '💎 VIP Skincare Enthusiasts': {
        'Email Strategy': 'Exclusive early access, VIP-only launches, personalized routines',
        'SMS/Push': 'Flash VIP sales, restock alerts for favorites, birthday luxe gifts',
        'Loyalty': 'Platinum perks, free samples with every order, concierge skincare advice',
        'Retention': 'Surprise & delight program, annual subscription incentives',
        'Budget': '30-35% of retention budget',
    },
    '⭐ Loyal Routine Buyers': {
        'Email Strategy': 'Replenishment reminders, routine upgrades, seasonal swaps',
        'SMS/Push': 'Reorder reminders, bundle discounts, referral program',
        'Loyalty': 'Gold tier benefits, points multipliers on bundles',
        'Retention': 'Subscription conversion campaigns, loyalty program upsell',
        'Budget': '20-25% of retention budget',
    },
    '🛒 Deal-Seeking Explorers': {
        'Email Strategy': 'Sale announcements, value bundles, price-drop alerts',
        'SMS/Push': 'Limited-time offers, clearance events',
        'Loyalty': 'Points for reviews, social shares for discounts',
        'Retention': 'Margin-positive bundles, free shipping thresholds',
        'Budget': '15-20% of retention budget',
    },
    '🌱 Casual Newcomers': {
        'Email Strategy': 'Welcome series, skincare education, quiz-based recommendations',
        'SMS/Push': 'First-purchase discount, app download incentive',
        'Loyalty': 'Easy Bronze tier wins, low-threshold rewards',
        'Retention': 'Second purchase campaign within 30 days',
        'Budget': '10-15% of retention budget',
    },
    '💤 Lapsed / Dormant': {
        'Email Strategy': 'Win-back sequence, "we miss you" campaigns, product reformulation news',
        'SMS/Push': 'Come-back coupon (20% off), new product alerts',
        'Loyalty': 'Points expiration reminder, bonus points to return',
        'Retention': 'Sunset after 3 failed win-back attempts; reduce frequency',
        'Budget': '5-10% of retention budget',
    },
    '🔬 Niche Product Fans': {
        'Email Strategy': 'Deep-dive ingredient content, new SKU alerts in their category',
        'SMS/Push': 'Category-specific launches, limited editions',
        'Loyalty': 'Expert status, early access to niche products',
        'Retention': 'Cross-category sampling, routine expansion',
        'Budget': '5-10% of retention budget',
    },
}

print('🎯 B2C CAMPAIGN PLAYBOOK')
print('=' * 70)
for persona, actions in playbook.items():
    if persona in unified['persona'].values:
        n = (unified['persona'] == persona).sum()
        print(f'\n{persona} ({n:,} customers)')
        print('-' * 55)
        for channel, action in actions.items():
            print(f'  {channel:18s}: {action}')

In [ ]:
# ============================================================
# STEP 9B: Executive Dashboard
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(18, 14))

# Revenue by persona
rev_by_seg = unified.groupby('persona')['total_revenue'].sum().sort_values()
rev_by_seg.plot.barh(ax=axes[0,0], color=plt.cm.Set2(np.linspace(0, 1, FINAL_K)), edgecolor='white')
axes[0,0].set_title('Total Revenue by Persona', fontweight='bold')
axes[0,0].set_xlabel('Revenue ($)')

# CLV vs Engagement
for p in unified['persona'].unique():
    m = unified['persona'] == p
    axes[0,1].scatter(unified.loc[m, 'engagement_index'],
                      unified.loc[m, 'estimated_clv'],
                      label=p[:12], s=8, alpha=0.3)
axes[0,1].set_title('CLV vs Engagement', fontweight='bold')
axes[0,1].set_xlabel('Engagement Index'); axes[0,1].set_ylabel('Estimated CLV ($)')
axes[0,1].legend(fontsize=7)

# Order frequency distribution
unified.boxplot(column='order_frequency', by='kmeans_cluster', ax=axes[1,0], showfliers=False)
axes[1,0].set_title('Order Frequency by Cluster', fontweight='bold')
plt.sca(axes[1,0]); plt.title('')

# Revenue share pie
rev_share = unified.groupby('persona')['total_revenue'].sum()
axes[1,1].pie(rev_share, labels=[p[:15] for p in rev_share.index],
              autopct='%1.1f%%', colors=plt.cm.Set3(np.linspace(0, 1, len(rev_share))),
              startangle=90)
axes[1,1].set_title('Revenue Share by Persona', fontweight='bold')

plt.suptitle('B2C Customer Segmentation — Executive Dashboard', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# STEP 9C: Export Segmented Customer File
# ============================================================
# Merge RFM segments
unified = unified.merge(rfm[['customer_id', 'RFM_Segment', 'RFM_Score']],
                        on='customer_id', how='left')

export_cols = [
    'customer_id', 'email', 'age_group', 'gender', 'city',
    'acquisition_channel', 'loyalty_tier', 'is_subscribed',
    'total_orders', 'total_revenue', 'estimated_clv', 'clv_tier',
    'engagement_index', 'purchase_intensity', 'order_frequency',
    'days_since_last_order', 'favorite_category',
    'kmeans_cluster', 'persona', 'gmm_cluster', 'gmm_confidence',
    'dbscan_cluster', 'RFM_Segment', 'RFM_Score'
]

export = unified[export_cols].copy()
export.to_csv('b2c_segmented_customers.csv', index=False)

print(f'✅ Exported {len(export):,} customers to b2c_segmented_customers.csv')
print(f'   Columns: {len(export_cols)}')

# Final summary
print(f'\n{"=" * 70}')
print(f'🎉 B2C SEGMENTATION COMPLETE')
print(f'{"=" * 70}')
print(f'  Customers segmented:  {len(unified):,}')
print(f'  Personas created:     {FINAL_K}')
print(f'  Methods applied:      RFM, K-Means, GMM, DBSCAN, Hierarchical')
print(f'  CLV tiers:            5')
print(f'  Total revenue:        ${unified["total_revenue"].sum():,.0f}')
print(f'  Top persona:          {unified.groupby("persona")["total_revenue"].sum().idxmax()}')
print(f'\n  Ready for Marketo campaign import & Salesforce sync.')